In [1]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 05_model_evaluation.ipynb
# =============================================================================

# Module 05: Unsupervised Anomaly Evaluation & Threat Explainability

### Unsupervised ML Framework Notice:
- **Unsupervised Anomaly Detection**: The 7 models (`Isolation Forest`, `One-Class SVM`, `LOF`, `Elliptic Envelope`, `PCA`, `DBSCAN`, `KMeans`) are trained without ground-truth labels on `employee_features.parquet`.
- **No Ground-Truth Labels**: Official CERT malicious-user target labels are not present in the behavioral feature dataset.
- **Evaluation Focus**: Model evaluation focuses on feature importance ranking, anomaly score distributions, model consensus agreement, and CERT Layer 2 behavioral pattern validation.

### System Pipeline Architecture:
ML Anomaly Detection (Layer 1) $\rightarrow$ Multi-Model Consensus $\rightarrow$ Risk Scoring $\rightarrow$ CERT Layer 2 Behavioral Pattern Validation

In [2]:
import warnings
from pathlib import Path

import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

print("Evaluation libraries loaded!")

Evaluation libraries loaded!


In [3]:
PROJECT_ROOT = Path("..").resolve()
FEATURE_FILE = PROJECT_ROOT / "datasets" / "features" / "employee_features.parquet"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
PLOT_DIR = PROJECT_ROOT / "plots"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# Load Scaler and Isolation Forest model
if (MODEL_DIR / "isolation_forest.pkl").exists():
    iso_forest = joblib.load(MODEL_DIR / "isolation_forest.pkl")
    scaler = joblib.load(MODEL_DIR / "scaler.pkl")
    print("Loaded Isolation Forest and Scaler artifacts successfully.")
else:
    iso_forest = None
    scaler = None
    print("Warning: Model pickle artifacts not found in models/.")

## 1. Tree-Based Feature Importance Ranking (Isolation Forest)

Feature importance is computed from the tree partition structures of the trained Isolation Forest estimators to identify top behavioral drivers behind anomaly flags.

In [4]:
# Extract Feature Importances from Isolation Forest estimator trees
if iso_forest is not None and hasattr(iso_forest, "estimators_"):
    # Calculate mean feature importances across all trees in isolation forest
    importances = np.mean([tree.tree_.compute_feature_importances(normalize=False) 
                           for tree in iso_forest.estimators_], axis=0)
    
    # Load feature column names
    if FEATURE_FILE.exists():
        df_feat = pd.read_parquet(FEATURE_FILE)
        ignore_cols = ['user', 'first_activity', 'last_activity', 'date', 'role', 'department']
        feature_names = [c for c in df_feat.columns if c not in ignore_cols]
    else:
        feature_names = [
            'after_hours_logon_count',
            'file_copy_count',
            'usb_connect_count',
            'email_external_count',
            'email_bcc_count',
            'http_job_search_count',
            'logon_count',
            'psychometric_N',
            'psychometric_O'
        ]
        
    feat_imp_df = pd.DataFrame({
        'Feature': feature_names[:len(importances)],
        'Importance': importances
    }).sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feat_imp_df.head(15), x='Importance', y='Feature', palette='viridis')
    plt.title("Top Behavioral Feature Drivers (Isolation Forest)", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PLOT_DIR / "feature_importance.png", dpi=300)
    plt.show()
    
    feat_imp_df.to_csv(REPORT_DIR / "feature_importances.csv", index=False)
    print("Feature importance report saved!")
else:
    print("Isolation Forest estimator tree details unavailable.")

## 2. CERT Layer 2 Behavioral Pattern Validation

**Behavioral Verification (Layer 2) Overview:**
- Independent behavioral consistency validation evaluating ML-flagged users against population-level statistical baselines (P90/P95 thresholds across 6 CERT dimensions: *Temporal*, *Device/USB*, *Multi-PC*, *Web*, *Email*, *Overall Volume*).
- **Note**: The historical 75.6% Layer 2 validation rate represents statistical behavioral evidence alignment, not supervised model accuracy.

In [5]:
# Unsupervised Model Characteristics Summary Table
model_char_data = {
    'Model': ['Isolation Forest', 'One-Class SVM', 'LOF', 'Elliptic Envelope', 'PCA Reconstruction', 'DBSCAN', 'K-Means Distance'],
    'Learning Paradigm': ['Unsupervised', 'Unsupervised', 'Unsupervised', 'Unsupervised', 'Unsupervised', 'Unsupervised', 'Unsupervised'],
    'Detection Technique': ['Random Partition Trees', 'RBF Hyperplane Boundary', 'Local Density Divergence', 'Gaussian Covariance', 'Reconstruction MSE', 'Density Clustering', 'Centroid Distance'],
    'Ground-Truth Required': ['No', 'No', 'No', 'No', 'No', 'No', 'No']
}

model_char_df = pd.DataFrame(model_char_data)
model_char_df.to_csv(REPORT_DIR / "model_characteristics.csv", index=False)
print("Unsupervised Model Suite Overview:")
model_char_df